In [1]:
!nvidia-smi

Sun May 19 09:19:37 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 530.30.02              Driver Version: 530.30.02    CUDA Version: 12.1     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                  Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf            Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-80GB           Off| 00000000:07:00.0 Off |                    0 |
| N/A   64C    P0              405W / 400W|  37686MiB / 81920MiB |    100%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [2]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [3]:
import torch
from transformers import Pix2StructProcessor, Pix2StructForConditionalGeneration, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"

model = Pix2StructForConditionalGeneration.from_pretrained("google/deplot").to(device)
processor = AutoProcessor.from_pretrained("google/deplot")

/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-05-19 09:19:41.618483: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-19 09:19:42.385226: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [4]:
from datasets import load_dataset

In [5]:
chartqa_test_ds = load_dataset("TeeA/ChartQA", split="test")
chartqa_test_ds

Dataset({
    features: ['id_image', 'image', 'table', 'chart_type', 'qa', 'vi_qa', 'vi_table'],
    num_rows: 1509
})

In [6]:
def preprocess_data_as_format(sample):
    sample['table'] = sample['table'].replace("<&>", "<0x0A>").replace("<|>", " | ")
    return sample

# vichartqa_test_ds = vichartqa_test_ds.map(preprocess_data_as_format)
chartqa_test_ds = chartqa_test_ds.map(preprocess_data_as_format)

In [8]:
from tqdm import tqdm

step = 10
for i in tqdm(range(160, len(chartqa_test_ds), step)):
    sample = chartqa_test_ds[i:i+step]
    image = sample['image']
    inputs = processor(images=image, text="Generate underlying data table of the figure below:", truncation=True,
                    padding='max_length', max_length=512, return_tensors="pt", add_special_tokens=True, max_patches=2048).to(device)

    generated_ids = model.generate(**inputs, max_new_tokens=1000)
    generated_caption = processor.batch_decode(generated_ids, skip_special_tokens=True)
    # for pred, groud in zip(generated_caption, sample['table']):
    # Mở tệp để ghi
    with open('benchmark_deplot.txt', 'a') as file:
        # Lặp qua các dự đoán và giá trị thực
        for pred, ground in zip(generated_caption, sample['table']):
            # Ghi từng dự đoán vào tệp
            file.write(pred + '\n')
            file.write('-' * 5 + '\n')
            # Ghi từng giá trị thực vào tệp
            file.write(ground + '\n')
            file.write('=' * 20 + '\n')

100%|██████████| 135/135 [3:00:46<00:00, 80.35s/it]  
